tool construct: answering questions about "Huawei's latest phone" --> search tool

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
# MY VERSION

from serpapi import SerpApiClient

def search(query: str) -> str:
    api_key = os.getenv("SERPAPI_API_KEY")

    params = {
        "engine" : "google",
        "q" : query,
        "api_key" : api_key,
        "gl": "cn",  
        "hl": "zh-cn", 
    }
    client = SerpApiClient(params)

    result = client.get_dict()


    # print(result.keys())
    # print(result.keys())
    # print(type(result["organic_results"]))
    # print(result["organic_results"][0].keys())
    # print(result["organic_results"][0]["title"])
    if "answer_box_list" in result:
        return "\n".join(result["answer_box_list"])

    if "answer_box" in result and "answer" in result["answer_box"]:
        return result["answer_box"]["answer"]

    if "knowledge_graph" in result:
        return result["knowledge_graph"]["description"]

    if "organic_results" in result:
        snippets = []
        for res in result["organic_results"][:3]:
            title = res.get("title", "")
            snippet = res.get("snippet", "")
            snippets.append(title + "/n" + snippet)

        return "\n\n".join(snippets)

    return "dont find out"

        



   



In [18]:
search("how long history does China have?")

'approximately 3,700 years'

In [ ]:
#Reference version
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    A practical web search engine tool based on SerpApi.
    It intelligently parses search results, prioritizing direct answers or knowledge graph information.
    """
    print(f"🔍 Executing [SerpApi] web search: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "Error: SERPAPI_API_KEY not configured in .env file."

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # Country code
            "hl": "zh-cn", # Language code
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # Intelligent parsing: prioritize finding the most direct answer
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # If no direct answer, return summaries of the first three organic results
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"Sorry, no information found about '{query}'."

    except Exception as e:
        return f"Error occurred during search: {e}"